# LeWM

> stable-worldmodel's LeWM: a JEPA world model planning in its latent space.

In [ ]:
#| default_exp wm.lewm.lewm

In [ ]:
#| hide
from nbdev.showdoc import *

**LeWM** encodes each frame into one embedding, and a causal transformer predicts the next
embedding from a window of past embeddings conditioned on the actions. It is trained in latent
space (next-embedding MSE + SIGReg, see `wm/04_train.ipynb`) and plans with `GoalMSE` towards the
encoded goal image. Same model and `rollout` as stable-worldmodel's, reading its images from
`obs_key` (raw ``uint8`` frames, preprocessed inside the model).

In [ ]:
#| export
# Adapted from stable-worldmodel's world models (`wm/`) (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import torch
from torch import nn

from stable_marl.wm.lewm.module import Embedder, ImagePreprocessor, MLP, Predictor, ViTEncoder

In [ ]:
#| export
class LeWM(nn.Module):
    """
    Parameters
    ----------
    encoder : nn.Module
        Images ``(N, C, h, w)`` -> embeddings ``(N, D)``
    predictor : Predictor
        Next embeddings from embeddings and action embeddings
    action_encoder : Embedder
        Actions ``(B, T, action_dim)`` -> ``(B, T, D)``
    projector, pred_proj : nn.Module, optional
        Heads applied to the encoder output and to the predictions
    preprocessor : nn.Module, optional
        Raw frames -> encoder input (e.g. :class:`ImagePreprocessor`)
    obs_key : str
        Info key of the frames, ``(B, T, ...)``
    """
    def __init__(self, encoder, predictor, action_encoder, projector=None, pred_proj=None, preprocessor=None,
                 obs_key: str = 'pixels'):
        super().__init__()
        self.encoder, self.predictor, self.action_encoder = encoder, predictor, action_encoder
        self.projector = projector or nn.Identity()
        self.pred_proj = pred_proj or nn.Identity()
        self.preprocessor = preprocessor or nn.Identity()
        self.obs_key = obs_key

    def encode(self, info: dict) -> dict:
        "Adds ``emb`` ``(B, T, D)`` (and ``act_emb`` if ``action`` is given)."
        frames = self.preprocessor(info[self.obs_key])
        B, T = frames.shape[:2]
        frames = frames.reshape(B * T, *frames.shape[2:]).to(next(self.encoder.parameters()).dtype)
        info['emb'] = self.projector(self.encoder(frames)).reshape(B, T, -1)
        if 'action' in info:
            info['act_emb'] = self.action_encoder(info['action'])
        return info

    def predict(self, emb: torch.Tensor, act_emb: torch.Tensor) -> torch.Tensor:
        "Next embeddings ``(B, T, D)`` of every position of ``emb`` ``(B, T, D)``, ``act_emb`` ``(B, T, A_emb)``."
        preds = self.predictor(emb, act_emb)
        B, T = preds.shape[:2]
        return self.pred_proj(preds.reshape(B * T, -1)).reshape(B, T, -1)

    def rollout(self, info: dict, action_sequence: torch.Tensor, history_size: int | None = None) -> dict:
        """
        Roll the candidates ``(B, S, T, action_dim)`` out from the ``H`` context frames
        ``info[obs_key]`` ``(B, S, H, ...)``; ``info['action_history']`` ``(B, S, H - 1, action_dim)``
        holds the actions executed between them (required when ``H > 1``). Adds ``predicted_emb``
        ``(B, S, H + T, D)``, whose first ``H`` entries are the encoded context (cached as ``emb``).
        """
        history_size = history_size or getattr(self.predictor, 'num_frames', 3)
        H = info[self.obs_key].size(2)
        B, S, T = action_sequence.shape[:3]
        act_past = info.get('action_history')
        if act_past is None:
            act_past = action_sequence.new_zeros(B, S, 0, action_sequence.size(-1))
        assert act_past.size(2) == H - 1, f"action_history must hold H - 1 = {H - 1} blocks, got {act_past.size(2)}"
        info['action'] = torch.cat([act_past, action_sequence[:, :, :1]], dim=2)
        if 'emb' not in info:   # encode the context once (no gradient into the encoder)
            init = self.encode({self.obs_key: info[self.obs_key][:, 0]})
            info['emb'] = init['emb'].detach().unsqueeze(1).expand(B, S, -1, -1)
        emb_init = info['emb'].reshape(B * S, *info['emb'].shape[2:])
        actions = torch.cat([act_past.reshape(B * S, *act_past.shape[2:]),
                             action_sequence.reshape(B * S, *action_sequence.shape[2:])], dim=1)
        all_act_emb = self.action_encoder(actions)   # index k: the block leaving frame k
        emb_list = list(emb_init.unbind(dim=1))
        for t in range(T):
            lo = max(0, H + t - history_size)
            emb_list.append(self.predict(torch.stack(emb_list[lo:], dim=1), all_act_emb[:, lo:H + t])[:, -1])
        info['predicted_emb'] = torch.stack(emb_list, dim=1).reshape(B, S, H + T, -1)
        return info

In [ ]:
#| export
def build_lewm(
    action_dim: int,
    image_size: int = 224,
    patch_size: int = 14,
    encoder_scale: str = 'tiny',
    embed_dim: int = 192,
    history_size: int = 3,
    depth: int = 6,
    heads: int = 16,
    dim_head: int = 64,
    mlp_dim: int = 2048,
    dropout: float = 0.1,
    projector_hidden: int = 2048,
    obs_key: str = 'pixels',
    encoder_kwargs: dict | None = None) -> LeWM:
    """
    A LeWM with stable-worldmodel's default configuration (ViT-tiny encoder, 6-layer predictor),
    for actions of size `action_dim` per planning step (``action_block * D``, or ``action_block * K``
    one-hot for discrete actions).
    """
    encoder = ViTEncoder.from_scale(encoder_scale, image_size=image_size, patch_size=patch_size, **(encoder_kwargs or {}))
    return LeWM(
        encoder=encoder,
        predictor=Predictor(num_frames=history_size, depth=depth, heads=heads, mlp_dim=mlp_dim, input_dim=embed_dim,
                            hidden_dim=embed_dim, output_dim=embed_dim, dim_head=dim_head, dropout=dropout),
        action_encoder=Embedder(input_dim=action_dim, emb_dim=embed_dim),
        projector=MLP(encoder.dim, projector_hidden, embed_dim, norm_fn=nn.BatchNorm1d),
        pred_proj=MLP(embed_dim, projector_hidden, embed_dim, norm_fn=nn.BatchNorm1d),
        preprocessor=ImagePreprocessor(image_size),
        obs_key=obs_key)

### Tests

In [ ]:
TINY = dict(image_size=32, patch_size=8, embed_dim=32, depth=2, heads=2, dim_head=16, mlp_dim=64, projector_hidden=64,
            encoder_kwargs=dict(dim=32, depth=2, heads=2, mlp_dim=64))
model = build_lewm(action_dim=7, history_size=3, **TINY).eval()
ctx = {'pixels': torch.randint(0, 256, (2, 4, 2, 56, 56, 3), dtype=torch.uint8),    # (B, S, H=2, h, w, C)
       'action_history': torch.randn(2, 4, 1, 7)}
out = model.rollout(ctx, torch.randn(2, 4, 5, 7))                                  # horizon 5
assert out['predicted_emb'].shape == (2, 4, 2 + 5, 32) and out['emb'].shape == (2, 4, 2, 32)
# the context embedding is cached: a second rollout reuses it
again = model.rollout(out, torch.randn(2, 4, 5, 7))
assert torch.equal(again['predicted_emb'][:, :, :2], out['predicted_emb'][:, :, :2])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()